# 04 · ML ranker

A model learns from 186 past allocations which available cornea was chosen. Logistic regression and LightGBM (LambdaMART) are compared; the best is chosen on training requests only. Code: `matching/features.py`, `matching/ranker.py`.

> **Privacy:** this notebook shows only counts, metrics, charts and study codes (R0001, Q0001), never NEBSL records. **Before committing, clear all outputs** (VS Code: `...` → *Clear All Outputs* → save).

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make `recommendation` importable whether this runs from notebooks/, recommendation/ or ml/
ML = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "recommendation" / "config.py").exists())
sys.path.insert(0, str(ML))

import pandas as pd
from IPython.display import Image, display
from recommendation.config import load_config

pd.set_option("display.width", 160)
cfg = load_config()
DATASET = cfg["paths"]["matching_dataset"]
OUTPUTS = Path(cfg["paths"]["outputs"])
print("Dataset:", Path(DATASET).name)

## 1. Features (23) and how they differ for chosen corneas

In [ ]:
from recommendation.matching.features import FEATURE_GROUPS, build_features
from recommendation.matching.mcda import Weights, score

pairs = pd.read_excel(DATASET, sheet_name="Ranking_Pairs")
X = build_features(pairs)
chosen = pairs["Label_Allocated"] == 1
pd.DataFrame({"group": {f: g for g, fs in FEATURE_GROUPS.items() for f in fs},
              "chosen": X[chosen].mean(), "not chosen": X[~chosen].mean(),
              "% missing": X.isna().mean() * 100}).round(2)

## 2. Correlation check

(a) each feature vs being chosen, compared within each request; (b) feature pairs that duplicate each other (|r| ≥ 0.8).

In [ ]:
Xc = X - X.groupby(pairs["Request_ID"]).transform("mean")
r_label = Xc.apply(lambda c: c.corr(chosen.astype(float))).sort_values(key=abs, ascending=False)
display(r_label.round(3).head(10).rename("r with chosen").to_frame())
C = X.corr()
dup = [(a, b, round(C.loc[a, b], 2)) for i, a in enumerate(C) for b in C.columns[i + 1:] if abs(C.loc[a, b]) >= 0.8]
pd.DataFrame(dup, columns=["feature", "feature", "r"]).sort_values("r", key=abs, ascending=False)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(C, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(C)), C.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(C)), C.columns, fontsize=7)
ax.set_title("Feature correlation (Pearson r)", loc="left", fontweight="bold")
fig.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout(); plt.show()

Highly correlated pairs don't harm ranking (the model is regularised) but make single coefficients unreliable, so explanations use SHAP grouped by clinical reason (notebook 05).

## 3. Model selection: 5-fold grouped CV on training requests only

In [ ]:
from recommendation.evaluation.split import grouped_split
from recommendation.matching.ranker import Ranker, repeated_cv_comparison, select

sp, m = cfg["split"], cfg["matching"]
scored = score(pairs, Weights(**m["weights"]))
train, test = grouped_split(scored, group_col=sp["group_col"], test_size=sp["test_size"], seed=sp["seed"])
best, table = select(train, seed=sp["seed"])
print("selected:", best)
table[["settings", "cv_top1", "cv_top3", "cv_mrr"]]

## 4. Held-out test requests

In [ ]:
from recommendation.evaluation.metrics import compare, ranking_metrics

model = Ranker(best).fit(train, sp["seed"])
ranked = model.rank(test)
res = compare(ranked)
res.loc["ML ranker"] = pd.Series(ranking_metrics(ranked, "ML_Rank"))
res.round(3)

## 5. Main result: all 186 requests, repeated grouped CV, with 95% CI

In [ ]:
rc = repeated_cv_comparison(scored, best, seed=sp["seed"])
display(pd.DataFrame(rc["methods"]).T.round(3))
pd.DataFrame(rc["difference"]).T.rename(columns={"ml_minus_mcda": "ML − MCDA"})

If the 95% CI includes 0, the ML ranker is not significantly better than MCDA on this data size.